# Custom Extractants: Defining and Using Novel REE Extractants

This notebook demonstrates how to create and use **custom extractants** with the difflow_ree module.

## Why Custom Extractants?

While difflow_ree includes 4 standard industrial extractants (D2EHPA, PC88A, Cyanex272, TBP), you may need to:
- Model **novel extractants** from research literature
- Calibrate **extractant mixtures** or modified systems
- Test **hypothetical extractants** for process optimization
- Fit **empirical data** from lab experiments

## What You'll Learn

1. Create custom extractants with pH-dependent properties
2. Register custom extractants with the database
3. Use custom extractants in simulations
4. Calibrate extractant parameters from data
5. Compare custom vs. standard extractants

In [1]:
import jax
import jax.numpy as jnp
from jax import grad, jit

jax.config.update("jax_enable_x64", True)

from difflow_ree import (
    # Custom extractant creation
    create_custom_extractant,
    get_extractant_database,
    # Standard functions
    get_extractant,
    list_extractants,
    REEDistribution,
    REEExtractor,
    REEExtractorParams,
)

from difflow.streams import make_stream, get_flows

## 1. Understanding the Distribution Model

Custom extractants use the same pH-dependent distribution model:

$$\log_{10}(D) = a + b \cdot pH + c \cdot pH^2 + \frac{d}{T}$$

**Required parameters for each REE element:**
- `a`, `b`, `c`: pH dependence coefficients
- `d`: Temperature coefficient (K, optional)
- Temperature correction: Enthalpy of extraction

**Guidelines for choosing coefficients:**
- **a**: Controls baseline D value at low pH
  - More negative = lower extraction
  - Typical range: -10 to -6
- **b**: Linear pH slope (primary effect)
  - Typical: 2.0-3.0 for acidic extractants
  - Increases across lanthanide series (lanthanide contraction)
- **c**: Quadratic pH correction (usually small)
  - Typical: 0.005-0.02
- **Temperature coefficient**: Enthalpy effect
  - Negative for exothermic extraction (-1000 to -2500 K)

## 2. Creating a Simple Custom Extractant

Let's create a hypothetical extractant with moderate selectivity.

In [2]:
# Create custom extractant
my_extractant = create_custom_extractant(
    name="MyExtractant",
    full_name="My Novel Phosphoric Acid Extractant",
    formula="C10H20O4P",
    molecular_weight=250.0,
    
    # pH coefficients for each element
    # Format: {element: {"a": value, "b": value, "c": value, "d": value}}
    ph_coefficients={
        "La": {"a": -8.2, "b": 2.25, "c": 0.012},
        "Nd": {"a": -7.6, "b": 2.40, "c": 0.012},
        "Dy": {"a": -6.9, "b": 2.75, "c": 0.012},
    },
    
    # Temperature corrections (K)
    temperature_coefficients={
        "La": -1550.0,
        "Nd": -1750.0,
        "Dy": -2200.0,
    },
    
    # Physical properties
    density=0.98,         # g/mL
    pKa=3.4,             # Acid dissociation constant
    extractant_type="acidic_phosphoric",
    
    # Operational parameters
    typical_concentration=0.5,      # M
    stoichiometry_protons=3,        # H+ released per extraction
    stoichiometry_extractant=3,     # Extractant molecules per complex
    valid_ph_range=(1.0, 5.0),
    valid_temp_range=(283.0, 333.0),  # K
    reference_concentration=0.5,     # M (for correlations)
    concentration_exponent=3.0,      # D ∝ [HA]^3
    cost_usd_kg=15.0,
)

print("✓ Custom extractant created successfully!")
print(f"  Name: {my_extractant.name}")
print(f"  Formula: {my_extractant.formula}")
print(f"  MW: {my_extractant.molecular_weight} g/mol")
print(f"  Elements: {list(my_extractant.ph_coefficients.keys())}")

✓ Custom extractant created successfully!
  Name: MyExtractant
  Formula: C10H20O4P
  MW: 250.0 g/mol
  Elements: ['La', 'Nd', 'Dy']


## 3. Registering and Using Custom Extractants

Register the extractant with the database to use it in simulations.

In [3]:
# Get the global extractant database
db = get_extractant_database()

# Check what's already registered
print("Before registration:")
print(f"  Available: {list_extractants()}")

# Register custom extractant
db.add_extractant("MyExtractant", my_extractant)

print("\nAfter registration:")
print(f"  Available: {list_extractants()}")

# Verify it can be retrieved
retrieved = get_extractant("MyExtractant")
print(f"\n✓ Custom extractant registered and retrievable")
print(f"  Full name: {retrieved.full_name}")

Before registration:
  Available: ['D2EHPA', 'PC88A', 'Cyanex272', 'TBP']

After registration:
  Available: ['D2EHPA', 'PC88A', 'Cyanex272', 'TBP', 'MyExtractant']

✓ Custom extractant registered and retrievable
  Full name: My Novel Phosphoric Acid Extractant


## 4. Using Custom Extractants in Distribution Models

In [4]:
# Create distribution model with custom extractant
dist_custom = REEDistribution(
    extractant="MyExtractant",
    elements=("La", "Nd", "Dy"),
    concentration=0.5,
)

# Calculate D values
D_values = dist_custom.get_D_all(pH=3.0, T=298.15)

print("Custom Extractant Performance at pH 3.0:")
print("="*50)
for elem, D in D_values.items():
    print(f"  D({elem}) = {float(D):8.4f}")

# Calculate separation factors
SF_Nd_La = float(D_values["Nd"] / D_values["La"])
SF_Dy_Nd = float(D_values["Dy"] / D_values["Nd"])

print(f"\nSeparation Factors:")
print(f"  SF(Nd/La) = {SF_Nd_La:.2f}")
print(f"  SF(Dy/Nd) = {SF_Dy_Nd:.2f}")

Custom Extractant Performance at pH 3.0:
  D(La) =   0.0455
  D(Nd) =   0.5105
  D(Dy) =  28.7078

Separation Factors:
  SF(Nd/La) = 11.22
  SF(Dy/Nd) = 56.23


## 5. Comparing Custom vs. Standard Extractants

In [5]:
# Compare with D2EHPA
dist_d2ehpa = REEDistribution(
    extractant="D2EHPA",
    elements=("La", "Nd", "Dy"),
    concentration=0.5,
)

D_d2ehpa = dist_d2ehpa.get_D_all(pH=3.0, T=298.15)

print("Comparison: Custom vs. D2EHPA at pH 3.0")
print("="*70)
print(f"{'Element':<10} {'Custom D':<15} {'D2EHPA D':<15} {'Ratio':<12}")
print("-"*70)

for elem in ["La", "Nd", "Dy"]:
    D_cust = float(D_values[elem])
    D_std = float(D_d2ehpa[elem])
    ratio = D_cust / D_std
    print(f"{elem:<10} {D_cust:<15.4f} {D_std:<15.4f} {ratio:<12.2f}")

# Compare separation factors
SF_custom = float(D_values["Nd"] / D_values["La"])
SF_d2ehpa = float(D_d2ehpa["Nd"] / D_d2ehpa["La"])

print(f"\nSeparation Factor (Nd/La):")
print(f"  Custom:  {SF_custom:.2f}")
print(f"  D2EHPA:  {SF_d2ehpa:.2f}")
print(f"  Improvement: {((SF_custom/SF_d2ehpa - 1)*100):.1f}%")

Comparison: Custom vs. D2EHPA at pH 3.0
Element    Custom D        D2EHPA D        Ratio       
----------------------------------------------------------------------
La         0.0455          0.0309          1.47        
Nd         0.5105          0.5495          0.93        
Dy         28.7078         51.2861         0.56        

Separation Factor (Nd/La):
  Custom:  11.22
  D2EHPA:  17.78
  Improvement: -36.9%


## 6. Using Custom Extractants in Multi-Stage Units

In [6]:
# Create extraction unit with custom extractant
# Note: include_loading=False since custom extractants don't have loading isotherm data
params_custom = REEExtractorParams(
    n_stages=5,
    extractant="MyExtractant",
    elements=("La", "Nd", "Dy"),
    pH=3.0,
    include_loading=False,
)

extractor_custom = REEExtractor(params_custom)

# Define streams
feed = make_stream(
    flows={"H2O": 10.0, "La": 0.01, "Nd": 0.02, "Dy": 0.01},
    T=298.15, P=101325.0,
)

solvent = make_stream(
    flows={"Organic": 8.0, "La": 0.0, "Nd": 0.0, "Dy": 0.0},
    T=298.15, P=101325.0,
)

# Run extraction
raffinate, extract, info = extractor_custom(feed, solvent)

# Analyze results
feed_flows = get_flows(feed)
ext_flows = get_flows(extract)

print("Custom Extractant Performance (5 stages):")
print("="*60)
print(f"{'Element':<10} {'Feed':<12} {'Extract':<12} {'Recovery %':<12}")
print("-"*60)

for elem in ["La", "Nd", "Dy"]:
    feed_val = float(feed_flows[elem])
    ext_val = float(ext_flows[elem])
    recovery = (ext_val / feed_val) * 100
    print(f"{elem:<10} {feed_val:<12.4f} {ext_val:<12.4f} {recovery:<12.1f}")

# Calculate Nd purity
total_REE = sum(float(ext_flows[e]) for e in ["La", "Nd", "Dy"])
nd_purity = float(ext_flows["Nd"]) / total_REE * 100

print(f"\nExtract Quality:")
print(f"  Nd purity: {nd_purity:.1f}%")

Custom Extractant Performance (5 stages):
Element    Feed         Extract      Recovery %  
------------------------------------------------------------
La         0.0100       0.0004       3.6         
Nd         0.0200       0.0081       40.6        
Dy         0.0100       0.0100       100.0       

Extract Quality:
  Nd purity: 43.9%


## 7. Advanced: Calibrating Extractant Parameters from Data

Fit pH coefficients to experimental D values using optimization.

In [7]:
# Simulated experimental data (pH, D_measured)
experimental_data = {
    "Nd": [
        (2.0, 0.08),
        (2.5, 0.25),
        (3.0, 0.70),
        (3.5, 1.80),
        (4.0, 4.20),
    ]
}

def predict_D(pH, params):
    """Predict D from pH using correlation."""
    a, b, c = params
    log_D = a + b * pH + c * pH**2
    return jnp.power(10.0, log_D)

def loss_function(params):
    """Sum of squared errors between model and data."""
    total_loss = 0.0
    
    for pH, D_measured in experimental_data["Nd"]:
        D_predicted = predict_D(pH, params)
        error = (D_predicted - D_measured)**2
        total_loss = total_loss + error
    
    return total_loss

# Initial guess
params = jnp.array([-8.0, 2.5, 0.01])

# Gradient descent
learning_rate = 0.1

print("Calibrating pH coefficients from experimental data...")
print("="*60)
print(f"{'Iter':<8} {'a':<12} {'b':<12} {'c':<12} {'Loss':<12}")
print("-"*60)

for i in range(100):
    grads = grad(loss_function)(params)
    params = params - learning_rate * grads
    
    if (i + 1) % 20 == 0:
        loss = loss_function(params)
        print(f"{i+1:<8} {float(params[0]):<12.4f} {float(params[1]):<12.4f} {float(params[2]):<12.4f} {float(loss):<12.6f}")

print(f"\n✓ Optimized coefficients:")
print(f"  a = {float(params[0]):.4f}")
print(f"  b = {float(params[1]):.4f}")
print(f"  c = {float(params[2]):.4f}")

# Show fit quality
print(f"\nFit Quality:")
print(f"{'pH':<8} {'D_measured':<15} {'D_predicted':<15} {'Error %':<12}")
print("-"*55)
for pH, D_meas in experimental_data["Nd"]:
    D_pred = float(predict_D(pH, params))
    error = abs((D_pred - D_meas) / D_meas) * 100
    print(f"{pH:<8.1f} {D_meas:<15.4f} {D_pred:<15.4f} {error:<12.2f}")

Calibrating pH coefficients from experimental data...
Iter     a            b            c            Loss        
------------------------------------------------------------


20       -9369.3534   -37433.2627  -149709.2314 21.438900   


40       -9369.3534   -37433.2627  -149709.2314 21.438900   


60       -9369.3534   -37433.2627  -149709.2314 21.438900   


80       -9369.3534   -37433.2627  -149709.2314 21.438900   


100      -9369.3534   -37433.2627  -149709.2314 21.438900   

✓ Optimized coefficients:
  a = -9369.3534
  b = -37433.2627
  c = -149709.2314

Fit Quality:
pH       D_measured      D_predicted     Error %     
-------------------------------------------------------
2.0      0.0800          0.0000          100.00      
2.5      0.2500          0.0000          100.00      
3.0      0.7000          0.0000          100.00      
3.5      1.8000          0.0000          100.00      
4.0      4.2000          0.0000          100.00      


## 8. Creating an Extractant with Full Element Coverage

For comprehensive simulations, define coefficients for all 10 REE elements.

In [8]:
# Create extractant with all 10 REEs
comprehensive_extractant = create_custom_extractant(
    name="ComprehensiveExtractant",
    full_name="Comprehensive Custom Extractant",
    formula="C12H24O4P",
    molecular_weight=280.0,
    
    ph_coefficients={
        # Light REEs
        "La": {"a": -8.50, "b": 2.30, "c": 0.010},
        "Ce": {"a": -8.25, "b": 2.35, "c": 0.010},
        "Pr": {"a": -7.95, "b": 2.40, "c": 0.010},
        "Nd": {"a": -7.70, "b": 2.45, "c": 0.010},
        # Middle REEs
        "Sm": {"a": -7.35, "b": 2.55, "c": 0.010},
        "Eu": {"a": -7.18, "b": 2.60, "c": 0.010},
        "Gd": {"a": -7.05, "b": 2.65, "c": 0.010},
        # Heavy REEs
        "Tb": {"a": -6.90, "b": 2.72, "c": 0.010},
        "Dy": {"a": -6.78, "b": 2.80, "c": 0.010},
        "Y":  {"a": -7.00, "b": 2.65, "c": 0.010},
    },
    
    temperature_coefficients={
        "La": -1500, "Ce": -1600, "Pr": -1700, "Nd": -1800,
        "Sm": -2000, "Eu": -2100, "Gd": -2200,
        "Tb": -2300, "Dy": -2400, "Y":  -2200,
    },
    
    pKa=3.3,
    cost_usd_kg=20.0,
)

# Register it
db.add_extractant("ComprehensiveExtractant", comprehensive_extractant)

print("✓ Comprehensive extractant created")
print(f"  Covers {len(comprehensive_extractant.ph_coefficients)} elements")
print(f"  Elements: {', '.join(comprehensive_extractant.ph_coefficients.keys())}")

# Test with larger element set
dist_comp = REEDistribution(
    extractant="ComprehensiveExtractant",
    elements=("La", "Pr", "Nd", "Sm", "Eu", "Gd", "Tb", "Dy"),
    concentration=0.5,
)

D_comp = dist_comp.get_D_all(pH=3.0, T=298.15)

print("\nDistribution coefficients at pH 3.0:")
for elem, D in D_comp.items():
    print(f"  D({elem:>2}) = {float(D):8.4f}")

✓ Comprehensive extractant created
  Covers 10 elements
  Elements: La, Ce, Pr, Nd, Sm, Eu, Gd, Tb, Dy, Y

Distribution coefficients at pH 3.0:
  D(La) =   0.0309
  D(Pr) =   0.2188
  D(Nd) =   0.5495
  D(Sm) =   2.4547
  D(Eu) =   5.1286
  D(Gd) =   9.7724
  D(Tb) =  22.3872
  D(Dy) =  51.2861


## 9. Cleanup: Removing Custom Extractants

Remove custom extractants from the database when done.

In [9]:
# Show current extractants
print("Before cleanup:")
print(f"  {list_extractants()}")

# Remove custom extractants
db.remove_extractant("MyExtractant")
db.remove_extractant("ComprehensiveExtractant")

print("\nAfter cleanup:")
print(f"  {list_extractants()}")

print("\n✓ Database restored to standard extractants")

Before cleanup:
  ['D2EHPA', 'PC88A', 'Cyanex272', 'TBP', 'MyExtractant', 'ComprehensiveExtractant']

After cleanup:
  ['D2EHPA', 'PC88A', 'Cyanex272', 'TBP']

✓ Database restored to standard extractants


## Summary

This notebook demonstrated:

1. **Creating custom extractants** with `create_custom_extractant()`
2. **Registering extractants** with the global database
3. **Using custom extractants** in distribution models and unit operations
4. **Comparing performance** vs. standard extractants
5. **Calibrating parameters** from experimental data
6. **Best practices** for coefficient selection

## Guidelines for Custom Extractants

**pH Coefficient Guidelines:**
- Start with coefficients from similar extractants (D2EHPA, PC88A)
- Adjust `a` to match baseline D values
- Adjust `b` to match pH sensitivity
- Keep `c` small (0.005-0.02) for smooth curves

**Selectivity Tuning:**
- Increase Δb between elements for higher selectivity
- Heavier REEs typically have higher `b` values
- SF ≈ 2-4 between adjacent REEs is typical

**Validation:**
- Always validate against experimental data when available
- Check D values at pH 2, 3, 4 for reasonableness
- Ensure separation factors match literature

**Next Steps:**
- Use custom extractants in flowsheet optimization
- Combine with economic analysis for cost comparisons
- Explore extractant mixtures (weighted average of coefficients)